# Phase 6 Naive CUDA Sobel (Google Colab)

Use this notebook in Google Colab with GPU runtime enabled. It creates the minimal project files, compiles the naive CUDA Sobel program, runs it on generated PGM inputs, and prints timing output for benchmark entry.

In [1]:
!nvidia-smi

Sat Mar 28 13:55:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from pathlib import Path

Path('include').mkdir(exist_ok=True)
Path('src/common').mkdir(parents=True, exist_ok=True)
Path('src/cuda').mkdir(parents=True, exist_ok=True)
Path('samples/images').mkdir(parents=True, exist_ok=True)

In [3]:
%%writefile include/image_io.hpp
#pragma once
#include <cstddef>
#include <cstdint>
#include <string>
#include <vector>

struct GrayImage {
    int width = 0;
    int height = 0;
    std::vector<std::uint8_t> pixels;

    bool empty() const;
    std::size_t size() const;
    std::uint8_t& at(int x, int y);
    const std::uint8_t& at(int x, int y) const;
};

GrayImage makeTestPattern(int width, int height);
GrayImage loadPgm(const std::string& path);
void savePgm(const GrayImage& image, const std::string& path);

Writing include/image_io.hpp


In [4]:
%%writefile src/common/image_io.cpp
#include "image_io.hpp"
#include <cctype>
#include <fstream>
#include <stdexcept>

namespace {
std::string readToken(std::istream& input) {
    std::string token;
    char ch = '\0';
    while (input.get(ch)) {
        if (std::isspace(static_cast<unsigned char>(ch))) {
            continue;
        }
        if (ch == '#') {
            input.ignore(4096, '\n');
            continue;
        }
        token.push_back(ch);
        break;
    }
    while (input.get(ch)) {
        if (std::isspace(static_cast<unsigned char>(ch))) {
            break;
        }
        token.push_back(ch);
    }
    if (token.empty()) throw std::runtime_error("Unexpected end of file while reading PGM token");
    return token;
}
}

bool GrayImage::empty() const { return pixels.empty() || width <= 0 || height <= 0; }
std::size_t GrayImage::size() const { return pixels.size(); }
std::uint8_t& GrayImage::at(int x, int y) { return pixels[static_cast<std::size_t>(y * width + x)]; }
const std::uint8_t& GrayImage::at(int x, int y) const { return pixels[static_cast<std::size_t>(y * width + x)]; }

GrayImage makeTestPattern(int width, int height) {
    GrayImage image;
    image.width = width;
    image.height = height;
    image.pixels.resize(static_cast<std::size_t>(width * height));
    for (int y = 0; y < height; ++y) {
        for (int x = 0; x < width; ++x) {
            std::uint8_t value = static_cast<std::uint8_t>((x + y) % 256);
            if (x > width / 4 && x < (3 * width) / 4 && y > height / 4 && y < (3 * height) / 4) value = 220;
            image.at(x, y) = value;
        }
    }
    return image;
}

GrayImage loadPgm(const std::string& path) {
    std::ifstream file(path, std::ios::binary);
    if (!file) throw std::runtime_error("Failed to open input image: " + path);
    if (readToken(file) != "P5") throw std::runtime_error("Only binary PGM (P5) is supported");
    GrayImage image;
    image.width = std::stoi(readToken(file));
    image.height = std::stoi(readToken(file));
    const int max_value = std::stoi(readToken(file));
    if (max_value != 255) throw std::runtime_error("Only 8-bit PGM files are supported");
    image.pixels.resize(static_cast<std::size_t>(image.width * image.height));
    file.read(reinterpret_cast<char*>(image.pixels.data()), static_cast<std::streamsize>(image.pixels.size()));
    if (!file) throw std::runtime_error("Failed to read image pixel data");
    return image;
}

void savePgm(const GrayImage& image, const std::string& path) {
    std::ofstream file(path, std::ios::binary);
    if (!file) throw std::runtime_error("Failed to open output image: " + path);
    file << "P5\n" << image.width << ' ' << image.height << "\n255\n";
    file.write(reinterpret_cast<const char*>(image.pixels.data()), static_cast<std::streamsize>(image.pixels.size()));
}

Writing src/common/image_io.cpp


In [5]:
%%writefile src/cuda/sobel_naive.cu
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include <cuda_runtime.h>
#include "image_io.hpp"

__global__ void sobelNaiveKernel(const std::uint8_t* input, std::uint8_t* output, int width, int height) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;
    int index = y * width + x;
    if (x == 0 || y == 0 || x == width - 1 || y == height - 1) { output[index] = 0; return; }
    int gx = -input[(y - 1) * width + (x - 1)] + input[(y - 1) * width + (x + 1)] - 2 * input[y * width + (x - 1)] + 2 * input[y * width + (x + 1)] - input[(y + 1) * width + (x - 1)] + input[(y + 1) * width + (x + 1)];
    int gy = -input[(y - 1) * width + (x - 1)] - 2 * input[(y - 1) * width + x] - input[(y - 1) * width + (x + 1)] + input[(y + 1) * width + (x - 1)] + 2 * input[(y + 1) * width + x] + input[(y + 1) * width + (x + 1)];
    int magnitude = min(255, abs(gx) + abs(gy));
    output[index] = static_cast<std::uint8_t>(magnitude);
}

void checkCuda(cudaError_t error, const char* step) {
    if (error != cudaSuccess) throw std::runtime_error(std::string(step) + ": " + cudaGetErrorString(error));
}

int main(int argc, char** argv) {
    try {
        if (argc != 3) { std::cerr << "Usage: ./sobel_naive <input.pgm> <output.pgm>\n"; return 1; }
        GrayImage input = loadPgm(argv[1]);
        GrayImage output;
        output.width = input.width;
        output.height = input.height;
        output.pixels.assign(input.size(), 0);
        std::size_t bytes = input.size() * sizeof(std::uint8_t);
        std::uint8_t *device_input = nullptr, *device_output = nullptr;
        cudaEvent_t total_start, total_stop, kernel_start, kernel_stop;
        checkCuda(cudaEventCreate(&total_start), "create total_start");
        checkCuda(cudaEventCreate(&total_stop), "create total_stop");
        checkCuda(cudaEventCreate(&kernel_start), "create kernel_start");
        checkCuda(cudaEventCreate(&kernel_stop), "create kernel_stop");
        checkCuda(cudaEventRecord(total_start), "record total_start");
        checkCuda(cudaMalloc(&device_input, bytes), "malloc input");
        checkCuda(cudaMalloc(&device_output, bytes), "malloc output");
        checkCuda(cudaMemcpy(device_input, input.pixels.data(), bytes, cudaMemcpyHostToDevice), "copy H2D");
        dim3 block_dim(16, 16);
        dim3 grid_dim((input.width + 15) / 16, (input.height + 15) / 16);
        checkCuda(cudaEventRecord(kernel_start), "record kernel_start");
        sobelNaiveKernel<<<grid_dim, block_dim>>>(device_input, device_output, input.width, input.height);
        checkCuda(cudaGetLastError(), "kernel launch");
        checkCuda(cudaEventRecord(kernel_stop), "record kernel_stop");
        checkCuda(cudaEventSynchronize(kernel_stop), "sync kernel_stop");
        checkCuda(cudaMemcpy(output.pixels.data(), device_output, bytes, cudaMemcpyDeviceToHost), "copy D2H");
        checkCuda(cudaEventRecord(total_stop), "record total_stop");
        checkCuda(cudaEventSynchronize(total_stop), "sync total_stop");
        float kernel_ms = 0.0f, total_ms = 0.0f;
        checkCuda(cudaEventElapsedTime(&kernel_ms, kernel_start, kernel_stop), "elapsed kernel");
        checkCuda(cudaEventElapsedTime(&total_ms, total_start, total_stop), "elapsed total");
        savePgm(output, argv[2]);
        std::cout << "Kernel time: " << kernel_ms << " ms\n";
        std::cout << "Total GPU path time: " << total_ms << " ms\n";
        cudaFree(device_input);
        cudaFree(device_output);
        cudaEventDestroy(total_start);
        cudaEventDestroy(total_stop);
        cudaEventDestroy(kernel_start);
        cudaEventDestroy(kernel_stop);
        return 0;
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << '\n';
        return 1;
    }
}

Writing src/cuda/sobel_naive.cu


In [6]:
%%writefile generate_pgm.py
from pathlib import Path

def write_pgm(path, width, height):
    pixels = bytearray(width * height)
    for y in range(height):
        for x in range(width):
            value = (x + y) % 256
            if x > width // 4 and x < (3 * width) // 4 and y > height // 4 and y < (3 * height) // 4:
                value = 220
            pixels[y * width + x] = value
    with open(path, 'wb') as f:
        f.write(f'P5\n{width} {height}\n255\n'.encode())
        f.write(pixels)

Path('samples/images').mkdir(parents=True, exist_ok=True)
for name, w, h in [('test_128x128.pgm', 128, 128), ('test_512x512.pgm', 512, 512), ('test_1280x720.pgm', 1280, 720), ('test_1920x1080.pgm', 1920, 1080)]:
    write_pgm(Path('samples/images') / name, w, h)
print('Generated benchmark inputs')

Writing generate_pgm.py


In [7]:
!python3 generate_pgm.py
!nvcc -O2 -I. -Iinclude src/cuda/sobel_naive.cu src/common/image_io.cpp -o sobel_naive

Generated benchmark inputs
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [18]:
!./sobel_naive samples/images/test_128x128.pgm samples/images/test_128x128_cuda_out.pgm
!./sobel_naive samples/images/test_512x512.pgm samples/images/test_512x512_cuda_out.pgm
!./sobel_naive samples/images/test_1280x720.pgm samples/images/test_1280x720_cuda_out.pgm
!./sobel_naive samples/images/test_1920x1080.pgm samples/images/test_1920x1080_cuda_out.pgm

Kernel time: 0.153696 ms
Total GPU path time: 0.365248 ms
Kernel time: 0.17056 ms
Total GPU path time: 0.484096 ms
Kernel time: 0.165824 ms
Total GPU path time: 0.883232 ms
Kernel time: 0.180768 ms
Total GPU path time: 1.55888 ms
